# Bayesian Inference & Simulation-Based Inference — Exercise Sheet

*Notation: $\theta$ denotes parameters, $x$ denotes data/observations, $p(\cdot)$ denotes a density.*

- Cluster 1 is a foundational exercise to a) derive a posterior when tractable and integrate analytically, and b) to numerically integrate or sample and approximate via Monte Carlo when a posterior is intractable. Good to do these things by hand once, especially if you want a more in-depth understanding through getting your hands dirty, but not necessary if you just want to work with SBI.

- 



---

## Cluster 1: Bayesian Foundations

### Exercise 1a — Gaussian Conjugate Posterior (by hand)

Consider a single observation $x$ drawn from a Gaussian likelihood with known variance $\sigma^2$:

$$p(x \mid \theta) = \mathcal{N}(x \mid \theta, \sigma^2)$$

and a Gaussian prior over the parameter $\theta$:

$$p(\theta) = \mathcal{N}(\theta \mid \mu_0, \tau_0^2)$$

**(a)** Derive the posterior $p(\theta \mid x)$ and show that it is Gaussian. Use the Gaussian density formula you saw in the lecture.

**(b)** Show that the posterior mean can be written as a precision-weighted average of the prior mean and the observation:

$$\mu_n = \frac{\frac{1}{\tau_0^2}\mu_0 + \frac{1}{\sigma^2}x}{\frac{1}{\tau_0^2} + \frac{1}{\sigma^2}}$$

**(c)** Generalize to $n$ i.i.d. observations $x_1, \dots, x_n$. What happens to the posterior as $n \to \infty$? Interpret this in terms of prior influence vanishing.

---

**Exercise 1b — Numerical Estimation of the Posterior Mean**

Consider inferring a neuron's firing rate $\lambda$ from an observed spike count $k$ over a fixed time window, using a Poisson likelihood:

$$p(k \mid \lambda) = \text{Poisson}(k \mid \lambda) = \frac{\lambda^k e^{-\lambda}}{k!}$$

and a Gaussian prior reflecting prior belief about this cell type's firing rate (e.g., from past literature):

$$p(\lambda) = \mathcal{N}(\lambda \mid \mu_0, \tau_0^2)$$

This pair is **not conjugate** — unlike the Gaussian-Gaussian case in Exercise 1a, there is no closed-form posterior here (the natural conjugate prior for a Poisson likelihood is a Gamma distribution).

**Naive rectangular grid quadrature.** Evaluate the unnormalized posterior $\tilde{p}(\lambda \mid k) = p(k\mid\lambda)\,p(\lambda)$ on an evenly-spaced grid $\{\lambda_1, \dots, \lambda_K\}$ over a suitable range, normalize via

$$p(\lambda_j \mid k) \approx \frac{\tilde{p}(\lambda_j \mid k)}{\sum_{i=1}^K \tilde{p}(\lambda_i \mid k)\,\Delta\lambda}$$

and estimate the posterior mean as a weighted sum. You can use `scipy.stats.poisson.pmf` and `scipy.stats.norm.pdf` (or `scipy.stats.poisson.logpmf` and `scipy.stats.norm.logpdf`) to evaluate the likelihood and prior, respectively.

Take $k=6$ spikes as your observation, and priors for the firing rates $\lambda \sim \mathcal{N}(5, 1^2)$ and $\lambda \sim \mathcal{N}(10, 1^2)$. Compare the posterior means and variances for these two priors.

Repeat for a coarser and a finer grid — how does accuracy scale with the number of grid points?

In [ ]:
from scipy import stats

k = 6
mu0_1 = 5
mu0_2 = 10
tau0 = 1

# del_lam = ... # define your grid of lambda 
prob_prior = stats.poisson.pmf(...) #
prob_lh = stats.norm.pdf(...) #
# sweep through the grid

TypeError: _parse_args() missing 1 required positional argument: 'mu'

---

## Group B: Conditional Density Estimation

### Exercise 2 — Implement a Mixture Density Network

Generate simulations from the toy simulator given below. Sample $N=500$ prior samples, $\theta \sim \text{Uniform}(-10, 10)$ and simulate their corresponding $x$ values.

**(a)** Fit a linear regression baseline predicting $\mathbb{E}[\theta \mid x]$. Note where it fails to capture the true conditional distribution (e.g., heteroscedasticity, multimodality).

**(b)** Implement a Mixture Density Network (MDN): a neural network $f_\phi(x) \to (\pi_k, \mu_k, \sigma_k)_{k=1}^K$ parameterizing a Gaussian mixture

$$p_\phi(\theta \mid x) = \sum_{k=1}^K \pi_k(x)\, \mathcal{N}\big(\theta \mid \mu_k(x), \sigma_k^2(x)\big)$$

Train by maximizing the log-likelihood of $\theta_i$ under $p_\phi(\theta \mid x_i)$.

**(c)** Visualize the predicted conditional density $p_\phi(\theta \mid x)$ at several fixed values of $x$, and compare against the regression baseline from (a). Where does the MDN correctly capture structure the baseline misses?



---

## Group C: Likelihood-Free Inference

### Exercise 3 — Rejection ABC

Consider a simple simulator $x = \text{sim}(\theta)$ with [1–2 free parameters TBD], from which we can sample but for which $p(x\mid\theta)$ has no tractable form.

**(a)** Generate a synthetic "observed" dataset $x_{\text{obs}} = \text{sim}(\theta^*)$ for some true $\theta^*$ (known to you, hidden from the exercise notionally).

**(b)** Implement rejection ABC:
1. Sample $\theta_i \sim p(\theta)$ from the prior
2. Simulate $x_i = \text{sim}(\theta_i)$
3. Accept $\theta_i$ if $d(x_i, x_{\text{obs}}) < \epsilon$ for some distance $d$ and tolerance $\epsilon$

Vary $\epsilon$ and report how the approximate posterior (the accepted $\theta_i$'s) changes in spread and bias.

**(c)** How many total simulations were needed to obtain, say, 500 accepted samples at your chosen $\epsilon$? Keep this number — you'll need it for Exercise 4.

---

### Exercise 4 — NPE vs. Rejection ABC: Simulation Efficiency

Using the same simulator and observed data from Exercise 3:

**(a)** Train a Neural Posterior Estimator (NPE) — a conditional density estimator $q_\phi(\theta \mid x)$ (e.g., your MDN from Exercise 2, or a provided normalizing flow) — on simulated pairs $\{(\theta_i, x_i)\}_{i=1}^N$ drawn from the prior.

**(b)** Evaluate $q_\phi(\theta \mid x_{\text{obs}})$ and compare its spread/accuracy to the rejection ABC posterior from Exercise 3, at a **matched total simulation budget** $N$.

**(c)** Plot posterior width (or an accuracy metric, if $\theta^*$ is known) as a function of simulation budget $N$ for both methods. At what budget does NPE start to outperform rejection ABC, if at all?

---

### Exercise 5 *(optional insert — Group C or D)* — Simulation-Based Calibration (SBC)

**(a)** For a **correctly specified** model (simulator matches the assumed generative process exactly), run SBC on your trained NPE from Exercise 4: repeatedly sample $\theta^{(m)} \sim p(\theta)$, simulate $x^{(m)}$, draw posterior samples from $q_\phi(\theta \mid x^{(m)})$, and compute the rank statistic of $\theta^{(m)}$ within those samples. Plot the rank histogram — it should be approximately uniform.

**(b)** Now introduce a deliberate misspecification: [TBD — e.g., wrong noise model, or simulator with an unmodeled additional parameter fixed at the wrong value]. Rerun SBC. How does the rank histogram deviate from uniform, and what does that deviation tell you about the direction/nature of the miscalibration?

---

### Exercise 6 *(optional insert — Group C or D)* — Amortization Mismatch

**(a)** Train NPE (as in Exercise 4) using a **deliberately narrow** prior $p(\theta)$ over the simulator's parameters.

**(b)** Query the trained posterior $q_\phi(\theta \mid x_{\text{obs}})$ on an observation $x_{\text{obs}}$ generated from a true $\theta^*$ **outside** the training prior's support.

**(c)** Describe what you observe. Does the posterior estimator fail visibly (e.g., flag low confidence) or silently (e.g., confidently wrong)? Why is this particularly dangerous in an amortized-inference setting compared to non-amortized methods?

---

## Group D: Real-World Application

### Exercise 7 — SBI on Real Intracellular Recordings

You are given intracellular recordings from [Allen Institute cell(s) TBD] under a step-current stimulation protocol.

**(a)** Implement (or use a provided implementation of) [an AdEx and/or Hodgkin-Huxley neuron model TBD], simulating voltage traces under the same step-current protocol as the real recording.

**(b)** Define summary statistics for both real and simulated traces: rheobase, spike count vs. injected current, inter-spike-interval (ISI) adaptation, and action potential width. [Additional features TBD.]

**(c)** Train an NPE using simulated $(\theta_i, x_i)$ pairs (summary statistics as $x$), then obtain the posterior $q_\phi(\theta \mid x_{\text{obs}})$ given the real cell's summary statistics.

**(d)** Run posterior predictive checks: simulate new traces using samples from $q_\phi(\theta \mid x_{\text{obs}})$ and compare their summary statistics against the real recording. On which features does the model reproduce the real cell well? Where does it visibly fail — and what does that tell you about model misspecification?

**(e)** *(If time/scope permits)* Repeat the fit using the other neuron model (AdEx vs. HH) on the same cell. Compare posterior spread and structure between the two models. Which model's posterior shows greater degeneracy/non-identifiability, and why might that be, given the relative parameter counts?

---

**A note on the optional inserts (Ex. 5, 6):** both are designed to be dropped into either Group C (using the toy simulator — failure mode is guaranteed and controllable) or Group D (using the real recording — failure mode is more authentic but depends on the specific cell/prior combination actually producing a legible failure; verify in advance if you choose this route).

---

This is a first pass — flag anything where the voice, framing, or level of scaffolding (e.g., how much is given vs. left for students) isn't matching what you want, and we can adjust before I turn it into the PDF.